# Phase A — Filtre de pertinence IT par embeddings (banc d'essai)

**But.** Valider, en local et hors Docker, qu'un modèle d'embeddings classe correctement
les objets d'appels d'offres en **IT** / **non-IT**, y compris les pièges « réseau de Maroua »,
« réseau d'adduction d'eau », « fourniture de matériel agricole ».

**Principe.** Pas de mots-clés. On encode des **phrases prototypes** IT et non-IT en deux
**centroïdes** ; un objet d'AO est classé selon le pôle dont il est le plus proche.

Exécute les cellules dans l'ordre (Maj+Entrée). La 1re fois, le modèle se télécharge une
seule fois puis reste en cache. Ce notebook ne touche à aucun conteneur.


## 0. Installation (à exécuter une seule fois)

In [1]:
# En local dans un venv, ou sur Google Colab / Kaggle.
%pip install -q sentence-transformers


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Charger le modèle d'embeddings

Modèle retenu : **mpnet** (meilleur que e5-base pour notre classification par centroïdes, cf. test comparatif).
`MODEL_NAME` reste une variable : tu peux comparer en changeant cette ligne.

| Modèle | Remarque |
|---|---|
| `paraphrase-multilingual-mpnet-base-v2` | **retenu** — précis, préfixes `query:`/`passage:` gérés |
| `paraphrase-multilingual-mpnet-base-v2` | alternative simple, plus léger |
| `BAAI/bge-m3` | excellent multilingue, plus lourd (textes longs) |

La cellule force un **cache local valide** : sur certaines machines une variable
d'environnement pointe vers un disque absent (ex. lecteur D:), ce qui provoque WinError 3.

In [2]:
import os, numpy as np

# Correctif cache : on force un dossier local valide AVANT tout chargement de modele.
# (Evite WinError 3 quand une variable d'env pointe vers un disque absent.)
# Si l'erreur persiste : menu Noyau > Redemarrer, puis re-executer depuis la cellule 0.
CACHE_DIR = os.path.join(os.path.expanduser("~"), ".cache", "kiama_models")
os.makedirs(CACHE_DIR, exist_ok=True)
for _v in ("HF_HOME", "HUGGINGFACE_HUB_CACHE", "TRANSFORMERS_CACHE", "SENTENCE_TRANSFORMERS_HOME"):
    os.environ[_v] = CACHE_DIR

from sentence_transformers import SentenceTransformer

MODEL_NAME = "paraphrase-multilingual-mpnet-base-v2"   # <- change ici pour comparer
model = SentenceTransformer(MODEL_NAME, cache_folder=CACHE_DIR)
print("Modele charge :", MODEL_NAME, "| cache:", CACHE_DIR)

_IS_E5 = "e5" in MODEL_NAME.lower()

def encode(texts, kind="passage"):
    """texte -> vecteur normalise. kind = 'query' (un AO) ou 'passage' (un prototype)."""
    if isinstance(texts, str):
        texts = [texts]
    if _IS_E5:
        prefix = "query: " if kind == "query" else "passage: "
        texts = [prefix + t for t in texts]
    v = model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)
    return v


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Modele charge : paraphrase-multilingual-mpnet-base-v2 | cache: C:\Users\Bafa'a\.cache\kiama_models


## 2. Définir les prototypes IT et non-IT

Deux pôles de phrases descriptives couvrant les grandes familles des portails d'Afrique
francophone (Cameroun, Côte d'Ivoire, Sénégal, Bénin, Congo). On décrit chaque famille du
côté négatif pour lever l'ambiguïté de « fourniture de matériel X ». Édite librement.

In [3]:
# Pole IT (positifs)
PROTO_IT = [
    "Développement, acquisition ou maintenance de logiciels et d'applications web et mobiles",
    "Conception et intégration de progiciels de gestion intégrée (ERP), SIRH, SIGFIP, SIGIPES",
    "Gestion électronique de documents (GED), archivage électronique et système d'information",
    "Fourniture et installation de matériel informatique : ordinateurs, serveurs, imprimantes, scanners, onduleurs",
    "Fourniture de consommables informatiques, périphériques et pièces détachées informatiques",
    "Équipement de salles informatiques et de laboratoires numériques",
    "Réseaux informatiques et télécommunications : LAN, fibre optique, câblage structuré, VSAT, interconnexion",
    "Téléphonie IP, visioconférence et équipements de télécommunication",
    "Cybersécurité, audit de sécurité, pare-feu et protection des systèmes d'information",
    "Cloud, datacenter, virtualisation, hébergement et infrastructure informatique",
    "Données, business intelligence, big data et intelligence artificielle",
    "E-gouvernement, dématérialisation, numérisation et plateformes en ligne",
    "Maintenance informatique, support technique et tierce maintenance applicative",
    "Solutions biométriques, cartes à puce, identification et contrôle d'accès informatisé",
    "Vidéosurveillance et systèmes électroniques de sécurité et de supervision",
]

# Pole non-IT (negatifs)
PROTO_NON_IT = [
    "Travaux de construction, génie civil et réhabilitation de bâtiments",
    "Construction et réhabilitation de routes, voirie, bitumage et ouvrages d'art",
    "Adduction d'eau potable, forages, assainissement et réseaux d'eau",
    "Électrification, réseau électrique, énergie solaire et éclairage public",
    "Matériel et intrants agricoles, semences, engrais et équipements ruraux",
    "Équipements et matériels médicaux, produits pharmaceutiques et consommables hospitaliers",
    "Mobilier de bureau, mobilier scolaire (tables-bancs) et fournitures de bureau",
    "Fourniture de matériels et équipements divers hors informatique",
    "Acquisition de véhicules, matériel roulant, engins, motos et pièces détachées",
    "Fourniture de carburant, lubrifiants et gaz",
    "Gardiennage, sécurité et surveillance",
    "Nettoyage, entretien des locaux et espaces verts",
    "Restauration, traiteur et fourniture de denrées alimentaires",
    "Impression, imprimerie, reprographie et édition",
    "Climatisation, froid, plomberie et électricité du bâtiment",
    "Confection d'uniformes, habillement et textile",
    "Études, formation, audit et assistance technique hors informatique",
    "Communication, événementiel, publicité et reportage",
]
print(len(PROTO_IT), "prototypes IT |", len(PROTO_NON_IT), "prototypes non-IT")


15 prototypes IT | 18 prototypes non-IT


## 3. Construire les deux centroïdes

In [4]:
def centroid(texts):
    v = encode(texts, kind="passage")
    c = v.mean(axis=0)
    return c / np.linalg.norm(c)

CEN_IT  = centroid(PROTO_IT)
CEN_NON = centroid(PROTO_NON_IT)
print("Centroides construits :", CEN_IT.shape)


Centroides construits : (768,)


## 4. La fonction de décision

Pertinent si l'objet est plus proche du pôle IT que du non-IT, d'une marge donnée (et
au-dessus d'un seuil minimal).

In [5]:
def is_relevant(objet, seuil=0.0, marge=0.02):
    v = encode(objet, kind="query")[0]
    sim_it  = float(v @ CEN_IT)
    sim_non = float(v @ CEN_NON)
    diff = sim_it - sim_non
    decision = (diff >= marge) and (sim_it >= seuil)
    return {"objet": objet, "sim_it": round(sim_it, 3),
            "sim_non": round(sim_non, 3), "diff": round(diff, 3),
            "pertinent": decision}

for o in ["Acquisition de materiel informatique pour les services centraux",
          "Travaux d'extension du reseau d'adduction d'eau potable de Maroua"]:
    print(is_relevant(o))


{'objet': 'Acquisition de materiel informatique pour les services centraux', 'sim_it': 0.758, 'sim_non': 0.53, 'diff': 0.228, 'pertinent': True}
{'objet': "Travaux d'extension du reseau d'adduction d'eau potable de Maroua", 'sim_it': 0.155, 'sim_non': 0.414, 'diff': -0.259, 'pertinent': False}


## 5. Jeu de test étiqueté (avec les pièges « réseau »)

In [6]:
TESTS = [
    ("Acquisition de materiel informatique : ordinateurs, imprimantes et onduleurs", True),
    ("Fourniture de materiels informatiques et reseaux pour les services centraux", True),
    ("Acquisition de licences de logiciels bureautiques (Adobe Acrobat Pro)", True),
    ("Mise en place d'une solution de dematerialisation des procedures administratives", True),
    ("Developpement d'une application web de gestion du courrier", True),
    ("Acquisition de serveurs et equipements de datacenter", True),
    ("Audit de cybersecurite du systeme d'information", True),
    ("Extension du reseau informatique local (LAN) et cablage structure", True),
    ("Travaux d'extension du reseau d'adduction d'eau potable de Maroua", False),
    ("Rehabilitation du reseau routier urbain de la ville de Garoua", False),
    ("Extension du reseau electrique et de l'eclairage public", False),
    ("Reseau de Maroua : etude d'amenagement urbain", False),
    ("Acquisition de petit materiel et outillage agricole", False),
    ("Fourniture de mobilier de bureau", False),
    ("Construction d'un cigare de stockage de GPL au depot petrolier", False),
    ("Fourniture de produits pharmaceutiques et consommables medicaux", False),
]
print(len(TESTS), "cas de test")


16 cas de test


## 6. Évaluer : tableau + précision / rappel / F1

In [7]:
def evaluate(seuil=0.0, marge=0.02, montrer=True):
    tp=fp=tn=fn=0; lignes=[]
    for objet, attendu in TESTS:
        r = is_relevant(objet, seuil, marge); pred = r["pertinent"]; ok = (pred == attendu)
        if pred and attendu: tp+=1
        elif pred and not attendu: fp+=1
        elif not pred and not attendu: tn+=1
        else: fn+=1
        lignes.append((("OK " if ok else "XX "), "IT " if pred else "non", r["sim_it"], r["sim_non"], r["diff"], objet[:54]))
    prec = tp/(tp+fp) if tp+fp else 0.0
    rapp = tp/(tp+fn) if tp+fn else 0.0
    f1 = 2*prec*rapp/(prec+rapp) if prec+rapp else 0.0
    if montrer:
        print(f"seuil={seuil}  marge={marge}")
        print(f"{'':3}{'pred':4}{'simIT':>7}{'simNON':>8}{'diff':>7}  objet")
        for l in lignes:
            print(f"{l[0]}{l[1]:4}{l[2]:7.3f}{l[3]:8.3f}{l[4]:7.3f}  {l[5]}")
        print(f"\nPrecision={prec:.2f}  Rappel={rapp:.2f}  F1={f1:.2f}  (TP={tp} FP={fp} TN={tn} FN={fn})")
        erreurs=[l[5] for l in lignes if l[0]=='XX ']
        print("Erreurs :", erreurs if erreurs else "aucune")
    return f1

_ = evaluate()


seuil=0.0  marge=0.02
   pred  simIT  simNON   diff  objet
OK IT    0.684   0.621  0.063  Acquisition de materiel informatique : ordinateurs, im
OK IT    0.760   0.653  0.107  Fourniture de materiels informatiques et reseaux pour 
OK IT    0.450   0.340  0.110  Acquisition de licences de logiciels bureautiques (Ado
OK IT    0.545   0.437  0.108  Mise en place d'une solution de dematerialisation des 
OK IT    0.566   0.379  0.187  Developpement d'une application web de gestion du cour
OK IT    0.587   0.386  0.201  Acquisition de serveurs et equipements de datacenter
OK IT    0.585   0.224  0.361  Audit de cybersecurite du systeme d'information
OK IT    0.580   0.355  0.225  Extension du reseau informatique local (LAN) et cablag
OK non   0.155   0.414 -0.259  Travaux d'extension du reseau d'adduction d'eau potabl
OK non   0.320   0.554 -0.233  Rehabilitation du reseau routier urbain de la ville de
OK non   0.471   0.564 -0.093  Extension du reseau electrique et de l'eclairage publi
OK n

## 7. Calibrer le seuil et la marge sur les données

In [8]:
best=None
for seuil in [-0.05, 0.0, 0.05, 0.10, 0.15]:
    for marge in [0.00, 0.02, 0.04, 0.06, 0.08, 0.10]:
        f1 = evaluate(seuil, marge, montrer=False)
        if best is None or f1 > best[0]:
            best=(f1, seuil, marge)
print(f"Meilleur : F1={best[0]:.2f}  seuil={best[1]}  marge={best[2]}\n")
_ = evaluate(best[1], best[2])


Meilleur : F1=1.00  seuil=-0.05  marge=0.0

seuil=-0.05  marge=0.0
   pred  simIT  simNON   diff  objet
OK IT    0.684   0.621  0.063  Acquisition de materiel informatique : ordinateurs, im
OK IT    0.760   0.653  0.107  Fourniture de materiels informatiques et reseaux pour 
OK IT    0.450   0.340  0.110  Acquisition de licences de logiciels bureautiques (Ado
OK IT    0.545   0.437  0.108  Mise en place d'une solution de dematerialisation des 
OK IT    0.566   0.379  0.187  Developpement d'une application web de gestion du cour
OK IT    0.587   0.386  0.201  Acquisition de serveurs et equipements de datacenter
OK IT    0.585   0.224  0.361  Audit de cybersecurite du systeme d'information
OK IT    0.580   0.355  0.225  Extension du reseau informatique local (LAN) et cablag
OK non   0.155   0.414 -0.259  Travaux d'extension du reseau d'adduction d'eau potabl
OK non   0.320   0.554 -0.233  Rehabilitation du reseau routier urbain de la ville de
OK non   0.471   0.564 -0.093  Extension du r

## 8. Validation sur AO réels (golden set)

Le F1 de la cellule 6 vaut pour des exemples choisis. On rejoue ici l'évaluation sur des
objets d'AO **réels** (ARMP et portails voisins) aux paramètres `seuil=-0.05, marge=0.0`.
Remplace/complète `GOLDEN` par TES objets copiés depuis armp.cm (True = IT). La sortie
signale une **zone grise** (|diff| < 0,05) : AO trop proches de la frontière, à vérifier.

In [9]:
SEUIL, MARGE = -0.05, 0.0

GOLDEN = [
    ("Acquisition de matériel informatique : cinquante ordinateurs de bureau, dix copieurs multifonctions et cinquante imprimantes laser (MINTP)", True),
    ("Fourniture de matériel informatique en trois lots distincts au profit de la Caisse Autonome d'Amortissement", True),
    ("Acquisition de matériel informatique au Port Autonome de Kribi", True),
    ("Acquisition de licences de logiciel spécifiques (Adobe Acrobat Pro et Xmind) - BCEAO", True),
    ("Fourniture de matériels informatiques et réseaux - AFRISTAT", True),
    ("Équipement des salles informatiques des établissements techniques et professionnels agricoles", True),
    ("Analyse, conception, développement et déploiement d'une plateforme de gestion des marchés publics", True),
    ("Fourniture et installation du matériel informatique à l'ANRP", True),
    ("Mise en place d'une solution de numérisation et d'archivage électronique des dossiers", True),
    ("Câblage et extension du réseau informatique local (LAN) du ministère", True),
    ("Acquisition du mobilier de bureau au profit des services centraux (MINFI)", False),
    ("Acquisition du petit matériel et outillage agricole (MINADER)", False),
    ("Acquisition de matériel roulant : quatre véhicules pick-up pour la production de biofertilisants (LANAVET)", False),
    ("Fourniture des matériels et travaux de construction d'un cigare de stockage de 250 TM de GPL (SCDP)", False),
    ("Fourniture des médicaments et dispositifs médicaux essentiels (CENAME)", False),
    ("Aménagement des bureaux de l'Institut National de la Statistique", False),
    ("Travaux d'extension du réseau d'adduction d'eau potable de Maroua", False),
    ("Réhabilitation du réseau routier urbain de la ville de Garoua", False),
    ("Extension du réseau électrique et de l'éclairage public", False),
]

def eval_golden(data, seuil=SEUIL, marge=MARGE, zone_grise=0.05):
    tp=fp=tn=fn=0; lignes=[]
    for objet, attendu in data:
        r=is_relevant(objet, seuil, marge); pred=r["pertinent"]; diff=r["diff"]; ok=(pred==attendu)
        flag = "!" if abs(diff)<zone_grise else " "
        if pred and attendu: tp+=1
        elif pred and not attendu: fp+=1
        elif (not pred) and (not attendu): tn+=1
        else: fn+=1
        lignes.append((("OK " if ok else "XX "), flag, "IT " if pred else "non", diff, objet[:60]))
    prec=tp/(tp+fp) if tp+fp else 0.0
    rapp=tp/(tp+fn) if tp+fn else 0.0
    f1=2*prec*rapp/(prec+rapp) if prec+rapp else 0.0
    print(f"{'':3}{'zg':3}{'pred':5}{'diff':>7}  objet")
    for l in lignes: print(f"{l[0]}{l[1]:3}{l[2]:5}{l[3]:7.3f}  {l[4]}")
    print(f"\nPrecision={prec:.2f}  Rappel={rapp:.2f}  F1={f1:.2f}  (TP={tp} FP={fp} TN={tn} FN={fn})")
    gris=[l[4] for l in lignes if l[1]=="!"]
    print("Zone grise |diff|<%.2f (a verifier) :"%zone_grise, gris if gris else "aucune")
    err=[l[4] for l in lignes if l[0]=="XX "]
    print("Erreurs :", err if err else "aucune")

eval_golden(GOLDEN)


   zg pred    diff  objet
OK    IT     0.080  Acquisition de matériel informatique : cinquante ordinateurs
OK    IT     0.144  Fourniture de matériel informatique en trois lots distincts 
OK    IT     0.137  Acquisition de matériel informatique au Port Autonome de Kri
OK    IT     0.128  Acquisition de licences de logiciel spécifiques (Adobe Acrob
OK    IT     0.206  Fourniture de matériels informatiques et réseaux - AFRISTAT
OK !  IT     0.042  Équipement des salles informatiques des établissements techn
OK !  IT     0.045  Analyse, conception, développement et déploiement d'une plat
OK    IT     0.158  Fourniture et installation du matériel informatique à l'ANRP
OK    IT     0.290  Mise en place d'une solution de numérisation et d'archivage 
OK    IT     0.193  Câblage et extension du réseau informatique local (LAN) du m
XX !  IT     0.034  Acquisition du mobilier de bureau au profit des services cen
OK    non   -0.255  Acquisition du petit matériel et outillage agricole (MINADER
OK 

### 8.1 Décision à 3 issues (zone grise = « à vérifier »)

On tranche seulement quand on est sûr ; sinon on délègue à l'humain. `eps` règle la largeur.

In [10]:
def classer(objet, eps=0.05):
    d = is_relevant(objet)["diff"]
    if d >=  eps:  return "IT"
    if d <= -eps:  return "non-IT"
    return "à vérifier"

for objet, _ in GOLDEN:
    print(f"{classer(objet):11} | {objet[:64]}")


IT          | Acquisition de matériel informatique : cinquante ordinateurs de 
IT          | Fourniture de matériel informatique en trois lots distincts au p
IT          | Acquisition de matériel informatique au Port Autonome de Kribi
IT          | Acquisition de licences de logiciel spécifiques (Adobe Acrobat P
IT          | Fourniture de matériels informatiques et réseaux - AFRISTAT
à vérifier  | Équipement des salles informatiques des établissements technique
à vérifier  | Analyse, conception, développement et déploiement d'une platefor
IT          | Fourniture et installation du matériel informatique à l'ANRP
IT          | Mise en place d'une solution de numérisation et d'archivage élec
IT          | Câblage et extension du réseau informatique local (LAN) du minis
à vérifier  | Acquisition du mobilier de bureau au profit des services centrau
non-IT      | Acquisition du petit matériel et outillage agricole (MINADER)
non-IT      | Acquisition de matériel roulant : quatre véhicules p

## 9. Conclusion et passage à la Phase B

- Si les pièges « réseau » sont écartés et les vrais IT gardés, le filtre est validé.
- Note le **modèle**, le **seuil/marge** et **eps** retenus : ce sont tes paramètres de production.
- **Phase B** : transposer `encode`, `centroid`, `is_relevant`/`classer` dans
  `decision_engine/src/semantic_filter.py`, derrière un drapeau `SEMANTIC_FILTER_ENABLED`,
  avec dégradation gracieuse, puis reconstruire le conteneur (Phase C).
